# Modelagem de dados — DuckDB interativo

**Objetivo:** explorar 4 hipóteses analíticas diretamente no Parquet da Fase 1 via DuckDB.  
Os números aqui se tornam benchmarks para validar o endpoint `/risk-summary` da Fase 3.

**Regra:** hipótese escrita em markdown *antes* do código — comparar previsão com resultado é o aprendizado.

In [ ]:
import duckdb

# Banco em memória — nada persiste no disco, lê o Parquet diretamente na query
con = duckdb.connect()
PARQUET = 'data/staging/credit_applications.parquet'
print('DuckDB', duckdb.__version__)

## Query 1 — Taxa de default por faixa etária

**Hipótese:** jovens (18-30) têm taxa de default maior que outras faixas porque têm histórico de crédito mais curto e renda mais baixa. A faixa 51+ deve ter a menor taxa — mais experiência financeira e ativos consolidados.

**Por que `AVG()` aqui?** `SeriousDlqin2yrs` é binário (0 ou 1). `AVG()` de uma coluna binária = proporção de 1s = taxa de default. Multiplico por 100 para ter percentual.

In [ ]:
df_age = con.execute(f"""
    SELECT
        CASE
            WHEN age BETWEEN 18 AND 30 THEN '18-30'
            WHEN age BETWEEN 31 AND 50 THEN '31-50'
            ELSE '51+'
        END AS age_group,
        ROUND(AVG(SeriousDlqin2yrs) * 100, 2) AS default_rate_pct,
        COUNT(*) AS total
    FROM '{PARQUET}'
    GROUP BY age_group
    ORDER BY age_group
""").df()

print(df_age.to_string(index=False))
# Resultado: 18-30 = 11.56%, 31-50 = 8.90%, 51+ = 4.38% — hipótese confirmada

## Query 2 — Renda mediana por status de default

**Hipótese:** inadimplentes têm renda mediana menor que adimplentes — restrição financeira é um fator causal clássico.

**Por que `MEDIAN()` e não `AVG()`?** A EDA da Fase 1 mostrou que P99 ($25k) está muito acima do P50 ($5.4k), com diferença de $1.270 entre média e mediana. A média seria distorcida pelos outliers de alta renda e não representaria o applicant típico.

In [ ]:
df_income = con.execute(f"""
    SELECT
        SeriousDlqin2yrs AS defaulted,
        MEDIAN(MonthlyIncome) AS median_income,
        COUNT(*) AS total
    FROM '{PARQUET}'
    GROUP BY defaulted
    ORDER BY defaulted
""").df()

print(df_income.to_string(index=False))
# Resultado: adimplentes $5.400, inadimplentes $5.240 — diferença pequena, hipótese confirmada

## Query 3 — Applicants com histórico severo de atrasos curtos

**Hipótese:** menos de 5% dos applicants têm mais de 3 atrasos de 30-59 dias.

**Nota sobre `SUM(COUNT(*)) OVER ()`:** com `WHERE` sem `GROUP BY`, a window cobre apenas o único grupo resultante — `pct_total` sempre retorna 100%. Para obter a proporção real do dataset total, usamos subquery com o total geral.

In [ ]:
# Versão correta: subquery para calcular % do total real
df_late = con.execute(f"""
    SELECT
        COUNT(*) AS n_applicants,
        ROUND(
            COUNT(*) * 100.0 / (SELECT COUNT(*) FROM '{PARQUET}'),
            2
        ) AS pct_of_total
    FROM '{PARQUET}'
    WHERE "NumberOfTime30-59DaysPastDueNotWorse" > 3
""").df()

print(df_late.to_string(index=False))
# Resultado: 1.597 applicants = ~1.06% do total — evento raro mas preditivo

## Query 4 — Correlação DebtRatio × default

**Hipótese:** correlação **positiva** e fraca (em torno de 0.05 a 0.15). DebtRatio alto sugere que uma pessoa gasta proporcionalmente mais do que ganha em dívidas — fator de risco.

**Por que 4 casas decimais?** Com N=150k, pequenas diferenças importam estatisticamente.

In [ ]:
df_corr = con.execute(f"""
    SELECT ROUND(CORR(DebtRatio, SeriousDlqin2yrs), 4) AS correlation
    FROM '{PARQUET}'
""").df()

print(df_corr.to_string(index=False))
# Resultado: -0.0076 — surpreendente! Correlação essencialmente zero e negativa.
# DebtRatio tem outliers extremos (valores > 100 no dataset bruto) que diluem qualquer sinal linear.
# Isso sugere que DebtRatio bruto não é um bom feature linear — transformação (log, clip) seria necessária.

## Sumário — benchmarks para o `/risk-summary` na Fase 3

| Métrica | Valor medido | Hipótese | Surpresa? |
|---------|-------------|----------|-----------|
| Default rate 18-30 | 11.56% | alta | não |
| Default rate 31-50 | 8.90% | média | não |
| Default rate 51+ | 4.38% | baixa | não |
| Renda mediana adimplentes | $5.400 | maior | não |
| Renda mediana inadimplentes | $5.240 | menor | não (diferença pequena) |
| % com > 3 atrasos curtos | 1.06% | < 5% | não |
| Correlação DebtRatio × default | -0.0076 | positiva e fraca | **sim** — essencialmente zero |

**Se o `/risk-summary` da Fase 3 divergir muito desses números → há um bug em alguma etapa de transformação.**

**Insight para o modelo ML (Fase futura):** DebtRatio bruto tem correlação quase zero com default — os outliers extremos (valores > 100) diluem o sinal. Antes de usar como feature, aplicar `clip(0, 5)` ou `log1p()` provavelmente revelaria correlação mais forte.